# 6.5 **池化层(Pooling)**

对于卷积操作，本质上是一个多层的映射，将信息逐渐降维聚合，映射为最后所需要表示的目标。若要搭建一个特地目标的模型，训练学习能够优化映射并最后保存“优势”。

之前提到过，卷积层的输出可以理解为一种“响应强度图”，卷积层本身具有局部性以及平移不变性，但是其输出的位置信息（空间分辨率）却很高，具有位置敏感性，即在哪个位置出现了卷积层所反映的模式。

但是，在现实当中，噪声存在（如图像因为扰动，像素模糊或者轻微偏移），位敏感度过高会带来信息冗余，比如对于其位置敏感性，我们想要的可能只是：这片区域存在这种模式。而卷积层输出带来了多余的信息。此外，对于卷积核在滑动过程中也会可能会忽略重要信息，或者由于轻微的噪声，导致重要像素点数据未被采样，输出因此受到干扰。

对此，我们需要一种更加模糊的映射或者说输出对位置敏感度更低的映射。即池化层。



## 6.5.1 MaxPooling and AveragePooling

与卷积层类似，池化层运算符由一个固定形状的窗口组成，该窗口根据其步幅大小在输入的所有区域上滑动，为固定形状窗口（有时称为汇聚窗口）遍历的每个位置计算一个输出。但池运算是确定性的（不需要学习，人为设定），我们通常计算汇聚窗口中所有元素的最大值或平均值。这些操作分别称为最大池化层（maximum pooling）和平均池化层（average pooling）。

e.g.

![alt text](https://zh-v2.d2l.ai/_images/pooling.svg)

原理：
$$
\begin{split}\max(0, 1, 3, 4)=4,\\
\max(1, 2, 4, 5)=5,\\
\max(3, 4, 6, 7)=7,\\
\max(4, 5, 7, 8)=8.\\\end{split}
$$

我们称此处的池化层为$2 \times 2$池化层，将该池化操作称为$2 \times 2$池化。（自然看得懂$p \times q$池化是什么意思了）

下面是代码实现，基础函数与[6.2](<6.2 图像卷积.ipynb>)中的corr2d函数相似。


In [1]:
import torch
from torch import nn
from d2l import torch as d2l

def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    h, w = X.shape
    Y = torch.zeros((h - p_h + 1, w - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i:i+p_h, j:j+p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i:i+p_h, j:j+p_w].mean()
    return Y

## 6.5.2 填充、步幅、多通道输入输出

同理卷积层，不多赘述。但输出通道是默认与输入通道相同的。

> **add**：默认情况下，深度学习框架中的步幅与汇聚窗口的大小相同。 因此，如果我们使用形状为(3, 3)的汇聚窗口，那么默认情况下，我们得到的步幅形状为(3, 3)。


In [2]:
#单通道
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
pool2d = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
pool2d(X)


tensor([[[[ 5.,  7.],
          [13., 15.]]]])

In [3]:
X = torch.cat((X, X + 1), 1)
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])

### 疑问与思考

1. 卷积核通过滑动来解决对细小变化敏感的问题，按理来说直接调整步幅即可，为什么还需要池化层呢？
2. 卷积层一般会和池化层同时出现吗？

ans

1. 通过滑动、调整步幅（Stride > 1）来解决，在逻辑上是可行的，这也是目前深度学习界的主流做法之一（比如 ResNet 就大量使用步幅为 2 的卷积来替代池化）。即卷积已经足够处理局部。且在特定需要精确位置信息的任务里池化就不是必要的了。
2. - 古典时期（AlexNet, VGG）： 几乎是雷打不动的“绑定”出现。通常是几个卷积层提取特征，紧接着一个池化层进行降维和抽象。
   - 现代时期（ResNet, EfficientNet）： 不再必然绑定。 如果你给网络足够的数据，网络自己可以通过带有步幅的卷积（Strided Convolution）学习到如何降维，这比人为设定“取最大值”这种死板的数学规则更灵活。现在很多先进的网络，整片架构中可能只在最后输出前使用一次全局平均池化（Global Average Pooling），中间的降维全部由步幅为 2 的卷积完成